# Fluxo experimental de classificação

## 1. Carregamento inicial
### 1.1 Importação das bibliotecas

Esta etapa verifica o arquivo, o esquema e a qualidade básica antes de qualquer modelagem. O objetivo é tornar explícito o que sabemos dos dados, não selecionar um modelo nem inferir significado clínico dos nomes codificados.

Coloque `final_treino_val.csv` em `data/raw/`. O arquivo é ignorado pelo Git. O regulamento define `desfecho=1` como classe positiva e `id` como identificador. Não use `id` como atributo preditor.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

import json
import hashlib
import sklearn
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold



In [ ]:
sns.set_theme(style="whitegrid")
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "final_treino_val.csv"
TARGET = "desfecho"
ID_COLUMN = "id"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {DATA_PATH.resolve()}. "
        "Copie o CSV local para data/raw/ sem adicioná-lo ao Git."
    )

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip().str.lower()
print(f"Dimensões: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print("Todos os registros rotulados participam do protocolo de validação cruzada.")


## 2. Preparação dos dados
### 2.1 Tratamento de inconsistências

Vamos confirmar a presença do identificador e do alvo, verificar os rótulos observados, valores ausentes, duplicatas e cardinalidade. Tipos inteiros, por si só, não significam que uma variável deva ser tratada como contínua; a classificação de tipos deve considerar o regulamento.

In [ ]:
required_columns = {ID_COLUMN, TARGET}
missing_required = required_columns - set(df.columns)
if missing_required:
    raise ValueError(f"Colunas obrigatórias ausentes: {sorted(missing_required)}")

print("Tipos de dados:")
display(df.dtypes.rename("dtype").to_frame())

quality = pd.Series(
    {
        "linhas": len(df),
        "colunas": df.shape[1],
        "linhas duplicadas": int(df.duplicated().sum()),
        "IDs ausentes": int(df[ID_COLUMN].isna().sum()),
        "IDs duplicados": int(df[ID_COLUMN].duplicated().sum()),
        "alvos ausentes": int(df[TARGET].isna().sum()),
    },
)
display(quality.rename("valor").to_frame())
print("Valores ausentes por coluna:")
display(df.isna().sum().sort_values(ascending=False).rename("ausentes").to_frame())

observed_target = set(df[TARGET].dropna().unique())
if not observed_target.issubset({0, 1}):
    raise ValueError(f"Rótulos inesperados em {TARGET}: {observed_target}")
print("Rótulos observados:", sorted(observed_target))

In [ ]:
feature_columns = [c for c in df.columns if c not in {ID_COLUMN, TARGET}]
cardinality = df[feature_columns].nunique(dropna=False).sort_values()
display(cardinality.rename("valores distintos").to_frame())

target_counts = df[TARGET].value_counts(dropna=False).sort_index()
target_summary = pd.DataFrame(
    {"contagem": target_counts, "proporcao": target_counts / len(df)}
)
display(target_summary)

ax = sns.countplot(data=df, x=TARGET, order=[0, 1], color="steelblue")
ax.set(title="Distribuição do desfecho", xlabel="Desfecho (0 = cura, 1 = óbito)", ylabel="Número de registros")
plt.tight_layout()
plt.show()

### 2.2 Criação das variáveis

Esta etapa registra a cardinalidade e estatísticas descritivas sem atribuir significado clínico aos códigos. Neste primeiro ciclo, não criaremos variáveis derivadas: o significado das categorias não foi divulgado e engenharia de atributos deve ser motivada e avaliada dentro do protocolo.

Uma diferença observada entre grupos é descritiva e não demonstra causalidade. Nenhuma decisão de modelagem ou limiar deve ser tomada com base no conjunto de teste final.

In [ ]:
display(df[feature_columns].describe(include="all").T)

# Proporções do alvo por valor observado; útil para inspeção, sem interpretação causal.
for column in feature_columns:
    table = pd.crosstab(df[column], df[TARGET], normalize="index")
    table.columns = [f"proporcao_desfecho_{value}" for value in table.columns]
    print(f"{column} (n valores={df[column].nunique(dropna=False)})")
    display(table)

### 2.3 Pré-processamento

Para os modelos, as variáveis nominais serão codificadas com one-hot encoding e as quantitativas tratadas numericamente. Qualquer imputação, codificação ou escala será ajustada dentro de um `Pipeline`/`ColumnTransformer` durante cada fold, evitando que informação dos folds de validação influencie o pré-processamento. A auditoria atual não encontrou ausências, mas os imputadores permitem tratar ausências se surgirem. Este pré-processador ainda não foi ajustado; na etapa 4 ele será combinado com cada estimador em um `Pipeline`, para que cada fold aprenda as transformações apenas no próprio treino.



In [ ]:
# Tipos definidos pelo regulamento e conferidos contra o esquema recebido.
binary_features = (
    [f"dadoclin{i}" for i in range(1, 15)]
    + ["sociodemografico1", "sociodemografico2"]
    + [f"comorbidade{i}" for i in range(1, 8)]
)
numeric_features = ["sociodemografico3"]
nominal_features = ["sociodemografico4", "sociodemografico5"]
expected_features = set(binary_features + numeric_features + nominal_features)
if set(feature_columns) != expected_features:
    raise ValueError("O esquema das variáveis preditoras difere do esperado; revisar antes de modelar.")

invalid_binary_values = {
    column: sorted(set(df[column].dropna().unique()) - {0, 1})
    for column in binary_features
    if not set(df[column].dropna().unique()).issubset({0, 1})
}
if invalid_binary_values:
    raise ValueError(f"Valores fora de 0/1 em variáveis binárias: {invalid_binary_values}")

binary_pipeline = SimpleImputer(strategy="most_frequent")
numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)
nominal_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("binary", binary_pipeline, binary_features),
        ("numeric", numeric_pipeline, numeric_features),
        ("nominal", nominal_pipeline, nominal_features),
    ],
    remainder="drop",
)
print("Pré-processador criado, ainda não ajustado.")


## 3. Protocolo de avaliação

**Estratégia:** validação cruzada estratificada aninhada em todos os registros rotulados, com 5 folds externos e 3 folds internos. Não há partição fixa reservada. Cada registro pode aparecer em folds de validação ao longo do desenvolvimento; por isso, a validação cruzada é uma estimativa interna de comparação, não um teste independente.

**Seleção e comparação:** ROC-AUC externa aninhada é o critério principal para comparar famílias. Em cada fold externo, a busca interna seleciona hiperparâmetros apenas no treino daquele fold. Ao final, uma nova busca estratificada de 5 folds seleciona os parâmetros usando todos os dados rotulados; o ajuste final também usa todos eles.

**Métricas:** reportar AUC externa por fold, média e dispersão descritiva. Métricas dependentes de limiar só devem ser usadas caso as regras oficiais exijam uma decisão binária e a regra de limiar seja definida a partir dos dados de desenvolvimento. A competição prioriza AUC-ROC.

**Prevenção de data leakage:** excluir `id` de `X`; manter imputação, codificação e escala dentro do `Pipeline` ajustado em cada fold; não usar o teste oficial nem seus rótulos para ajustar transformações, treinar, selecionar modelo/parâmetros, calibrar scores ou escolher limiar. Se o teste tiver `DESFECHO`, calcular métricas somente depois de produzir e salvar scores.


In [ ]:
predictor_columns = binary_features + numeric_features + nominal_features
X_dev = df[predictor_columns].copy()
y_dev = df[TARGET].astype(int).copy()
ids = df[ID_COLUMN].copy()  # Guardado à parte para rastreio e submissão.

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Registros rotulados em CV: {len(X_dev):,}")
print("Preditores:", X_dev.shape[1], "| identificador fora de X:", ID_COLUMN not in X_dev.columns)


## 4. Spot-checking dos algoritmos

### 4.1 Definição dos modelos e hiperparâmetros

Vamos comparar seis famílias em uma busca limitada e pré-definida: regressão logística como referência; Random Forest e Extra Trees para bagging; Gradient Boosting e HistGradientBoosting para boosting; e SVC com kernel RBF como método baseado em margem. Os ensembles de árvores e o SVC são candidatos capazes de capturar relações não lineares; essa é uma hipótese para experimentar, não uma garantia de melhor AUC.

A busca aleatória usa listas explícitas de valores, `scoring="roc_auc"` e semente fixa. Serão testadas 8 configurações em cada busca interna da validação aninhada e 15 na busca final em todos os registros rotulados. O número de árvores fica fixo em 400 para Random Forest e Extra Trees, evitando gastar orçamento em um parâmetro cuja função principal é estabilizar o ensemble.

**Controle de overfitting:** a validação externa estratificada de 5 folds estima o desempenho de cada família; em cada fold externo, uma busca interna estratificada de 3 folds seleciona hiperparâmetros usando apenas o treino externo. Esse desenho separa seleção e avaliação e reduz o viés otimista de reportar a mesma CV usada para escolher hiperparâmetros. Depois, uma nova busca final seleciona parâmetros em todos os registros rotulados. Não existe holdout fixo.

In [ ]:
models = {
    "logistic_regression": LogisticRegression(
        solver="lbfgs", l1_ratio=0.0, max_iter=3000, random_state=42
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=400, random_state=42, n_jobs=1
    ),
    "extra_trees": ExtraTreesClassifier(
        n_estimators=400, random_state=42, n_jobs=1
    ),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        random_state=42, early_stopping=False
    ),
    "svc_rbf": SVC(kernel="rbf"),
}

search_spaces = {
    "logistic_regression": {
        "model__C": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0],
        "model__class_weight": [None, "balanced"],
    },
    "random_forest": {
        "model__max_depth": [None, 6, 10, 16],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4, 8, 16],
        "model__max_features": ["sqrt", 0.5, 0.8, 1.0],
        "model__class_weight": [None, "balanced_subsample"],
    },
    "extra_trees": {
        "model__max_depth": [None, 6, 10, 16],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4, 8, 16],
        "model__max_features": ["sqrt", 0.5, 0.8, 1.0],
        "model__class_weight": [None, "balanced"],
    },
    "gradient_boosting": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [1, 2, 3],
        "model__min_samples_leaf": [5, 10, 20],
        "model__subsample": [0.7, 0.9, 1.0],
    },
    "hist_gradient_boosting": {
        "model__max_iter": [100, 200, 300, 500],
        "model__learning_rate": [0.03, 0.05, 0.08, 0.1],
        "model__max_leaf_nodes": [7, 15, 31],
        "model__min_samples_leaf": [10, 20, 30, 50],
        "model__l2_regularization": [0.0, 0.1, 1.0, 5.0],
    },
    "svc_rbf": {
        "model__C": [0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0],
        "model__gamma": ["scale", 0.001, 0.01, 0.1],
        "model__class_weight": [None, "balanced"],
    },
}

assert set(models) == set(search_spaces)
print(f"Famílias: {len(models)} | configurações por busca interna: 8 | busca final: 15")

### 4.2 Execução dos algoritmos

Cada busca recebe uma `Pipeline` nova com o pré-processador. O `RandomizedSearchCV` clona e ajusta a pipeline dentro de cada fold, mantendo imputação, escala e one-hot encoding restritos ao treino daquele fold. A AUC externa é calculada em observações que não participaram nem do ajuste nem da escolha dos hiperparâmetros daquele fold.

A tabela de AUC aninhada é a comparação menos enviesada entre as famílias. A AUC da busca final nos dados de desenvolvimento serve apenas para escolher parâmetros e não deve ser apresentada como desempenho final. O SVC usa `decision_function` para calcular AUC; se for selecionado, a etapa de submissão deverá calibrar probabilidades usando apenas desenvolvimento. A validação aninhada não é um teste independente.

In [ ]:
N_ITER_NESTED = 8
N_ITER_FINAL = 15
INNER_SPLITS = 3
outer_cv = cv  # 5 folds estratificados, random_state=42, definidos no protocolo.

nested_scores = {name: [] for name in models}
nested_fold_params = {name: [] for name in models}

for model_name, model in models.items():
    print(f"Validação aninhada: {model_name}", flush=True)
    for fold_number, (train_idx, valid_idx) in enumerate(outer_cv.split(X_dev, y_dev), start=1):
        X_outer_train = X_dev.iloc[train_idx]
        y_outer_train = y_dev.iloc[train_idx]
        X_outer_valid = X_dev.iloc[valid_idx]
        y_outer_valid = y_dev.iloc[valid_idx]

        inner_cv = StratifiedKFold(
            n_splits=INNER_SPLITS, shuffle=True, random_state=100 + fold_number
        )
        pipeline = Pipeline(
            steps=[("preprocessor", clone(preprocessor)), ("model", clone(model))]
        )
        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=search_spaces[model_name],
            n_iter=N_ITER_NESTED,
            scoring="roc_auc",
            cv=inner_cv,
            refit=True,
            random_state=42 + fold_number,
            n_jobs=-1,
            error_score="raise",
            return_train_score=False,
        )
        search.fit(X_outer_train, y_outer_train)

        fitted = search.best_estimator_
        if hasattr(fitted, "predict_proba"):
            outer_scores = fitted.predict_proba(X_outer_valid)[:, 1]
        else:
            outer_scores = fitted.decision_function(X_outer_valid)
        nested_scores[model_name].append(
            float(roc_auc_score(y_outer_valid, outer_scores))
        )
        nested_fold_params[model_name].append(
            {key: value.item() if hasattr(value, "item") else value
             for key, value in search.best_params_.items()}
        )
        print(f"  fold {fold_number}: AUC externa={nested_scores[model_name][-1]:.4f}", flush=True)

nested_summary = pd.DataFrame(
    [
        {
            "modelo": name,
            "auc_externa_media": np.mean(scores),
            "auc_externa_dp": np.std(scores, ddof=1),
            "auc_por_fold": scores,
        }
        for name, scores in nested_scores.items()
    ]
).sort_values("auc_externa_media", ascending=False).reset_index(drop=True)

display(nested_summary)

# Busca final dos hiperparâmetros em todos os registros rotulados.
final_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
final_searches = {}
for model_name, model in models.items():
    print(f"Busca final em 100% dos rótulos: {model_name}", flush=True)
    pipeline = Pipeline(
        steps=[("preprocessor", clone(preprocessor)), ("model", clone(model))]
    )
    final_search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=search_spaces[model_name],
        n_iter=N_ITER_FINAL,
        scoring="roc_auc",
        cv=final_cv,
        refit=True,
        random_state=2026,
        n_jobs=-1,
        error_score="raise",
        return_train_score=False,
    )
    final_search.fit(X_dev, y_dev)
    final_searches[model_name] = final_search
    print(f"  melhor AUC de seleção CV={final_search.best_score_:.4f}", flush=True)

stage4_results = {
    "protocol": {
        "scikit_learn_version": sklearn.__version__,
        "positive_class": 1,
        "selection_metric": "roc_auc",
        "outer_cv": {"type": "StratifiedKFold", "n_splits": 5, "shuffle": True, "random_state": 42},
        "inner_cv": {"type": "StratifiedKFold", "n_splits": INNER_SPLITS, "shuffle": True, "random_state": "100 + outer fold"},
        "nested_random_search_iterations_per_fold": N_ITER_NESTED,
        "final_development_search_iterations": N_ITER_FINAL,
        "labeled_rows_used": int(len(X_dev)),
        "data_scope": "all labeled rows; no fixed holdout",
        "labeled_training_file_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
        "preprocessing_inside_pipeline": True,
    },
    "nested_cv": {
        name: {
            "outer_auc_by_fold": scores,
            "outer_auc_mean": float(np.mean(scores)),
            "outer_auc_std": float(np.std(scores, ddof=1)),
            "best_params_by_outer_fold": nested_fold_params[name],
        }
        for name, scores in nested_scores.items()
    },
    "best_params_on_all_labeled_data": {
        name: {
            "best_cv_auc_for_selection_only": float(search.best_score_),
            "best_params": {
                key: value.item() if hasattr(value, "item") else value
                for key, value in search.best_params_.items()
            },
        }
        for name, search in final_searches.items()
    },
    "search_spaces": search_spaces,
}

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_PATH = RESULTS_DIR / "stage4_model_search.json"
RESULTS_PATH.write_text(
    json.dumps(stage4_results, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Resultados e hiperparâmetros salvos em: {RESULTS_PATH}")

## 5. Resultados

A comparação entre famílias usa exclusivamente as AUCs externas da validação aninhada calculada sobre todos os registros rotulados. Os folds compartilham registros de treino e validação ao longo do processo; portanto, os resultados são estimativas internas de desenvolvimento e não um teste independente.

### 5.1 Tabelas comparativas

A primeira tabela compara famílias pela AUC externa aninhada; a segunda documenta parâmetros selecionados pela busca final em todos os rótulos. A AUC dessa busca final é apenas critério de seleção, não uma estimativa independente de desempenho.


In [ ]:
stage4_path = PROJECT_ROOT / "results" / "stage4_model_search.json"
stage4_results = json.loads(stage4_path.read_text(encoding="utf-8"))

nested_rows = []
for model_name, result in stage4_results["nested_cv"].items():
    fold_aucs = result["outer_auc_by_fold"]
    nested_rows.append({"Modelo": model_name, "AUC externa média": result["outer_auc_mean"],
                        "DP entre folds": result["outer_auc_std"], "AUC mínima": min(fold_aucs),
                        "AUC máxima": max(fold_aucs)})
nested_table = pd.DataFrame(nested_rows).sort_values("AUC externa média", ascending=False).reset_index(drop=True)
display(nested_table.round(4))

parameter_rows = []
for model_name, result in stage4_results["best_params_on_all_labeled_data"].items():
    parameter_rows.append({"Modelo": model_name,
        "AUC CV para seleção (não final)": result["best_cv_auc_for_selection_only"],
        "Melhores parâmetros": json.dumps(result["best_params"], ensure_ascii=False)})
parameter_table = pd.DataFrame(parameter_rows).sort_values("AUC CV para seleção (não final)", ascending=False).reset_index(drop=True)
display(parameter_table.round({"AUC CV para seleção (não final)": 4}))


### 5.2 Gráficos da validação aninhada

Os gráficos mostram médias, dispersão descritiva e valores por fold da validação externa. Eles não representam teste independente nem intervalo de confiança.


In [ ]:
# 1) Médias da AUC externa com dispersão descritiva entre folds.
plot_table = nested_table.sort_values("AUC externa média", ascending=True).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8.5, 5.2))
y_positions = np.arange(len(plot_table))
ax.errorbar(
    plot_table["AUC externa média"], y_positions,
    xerr=plot_table["DP entre folds"], fmt="o", capsize=4,
    color="#1f4e79", ecolor="#7f9db9", markersize=6,
)
ax.set_yticks(y_positions, plot_table["Modelo"].str.replace("_", " "))
ax.set_xlim(0.70, 0.78)
ax.set_xlabel("AUC externa média (eixo exibido: 0,70–0,78)")
ax.set_title("Comparação de modelos: média ± 1 DP entre folds")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "results" / "stage5_nested_auc_mean_sd.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

# 2) Scores brutos dos folds, exibidos como pontos sem impor ordem entre folds.
fig, ax = plt.subplots(figsize=(9, 5.2))
model_names = plot_table["Modelo"].tolist()
base_positions = np.arange(len(model_names))
fold_offsets = np.linspace(-0.20, 0.20, 5)
fold_colors = sns.color_palette("colorblind", n_colors=5)
for fold_index, (offset, color) in enumerate(zip(fold_offsets, fold_colors), start=1):
    fold_values = [
        stage4_results["nested_cv"][name]["outer_auc_by_fold"][fold_index - 1]
        for name in model_names
    ]
    ax.scatter(fold_values, base_positions + offset, color=color, s=38,
               label=f"Fold {fold_index}", zorder=3)
ax.set_yticks(base_positions, [name.replace("_", " ") for name in model_names])
ax.set_xlim(0.70, 0.78)
ax.set_xlabel("ROC-AUC externa (eixo exibido: 0,70–0,78)")
ax.set_title("AUC externa observada em cada fold")
ax.grid(axis="x", alpha=0.25)
ax.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.tight_layout(rect=(0, 0, 0.82, 1))
fig.savefig(PROJECT_ROOT / "results" / "stage5_nested_auc_by_fold.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)



## 6. Interpretação dos resultados

### 6.1 Leitura da comparação

Use a tabela e os gráficos produzidos pela execução atual. Diferenças pequenas entre médias e dispersão entre folds pedem cautela. Os resultados são de validação cruzada aninhada interna e podem orientar a seleção, mas não constituem uma avaliação independente após comparar famílias.

### 6.2 Limites

As conclusões se limitam às variáveis, famílias, espaços de busca e sementes especificados neste notebook. A importância preditiva por permutação é associativa, não causal. Confirme também se cada preditor estará disponível no momento da previsão.


In [ ]:
from sklearn.inspection import permutation_importance

importance_model_name = max(
    stage4_results["nested_cv"],
    key=lambda name: stage4_results["nested_cv"][name]["outer_auc_mean"],
)  # selecionado antes de pontuar o teste oficial
importance_nested = stage4_results["nested_cv"][importance_model_name]
importance_by_fold = []
importance_fold_auc = []

# As importâncias são calculadas somente em folds externos do desenvolvimento.
# Permutar a entrada original avalia cada coluna nominal como um conjunto de dummies.
for fold_number, (train_idx, valid_idx) in enumerate(cv.split(X_dev, y_dev)):
    importance_pipeline = Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", clone(models[importance_model_name])),
        ]
    )
    importance_pipeline.set_params(
        **importance_nested["best_params_by_outer_fold"][fold_number]
    )
    importance_pipeline.fit(X_dev.iloc[train_idx], y_dev.iloc[train_idx])
    X_valid = X_dev.iloc[valid_idx]
    y_valid = y_dev.iloc[valid_idx]

    fold_scores = importance_pipeline.predict_proba(X_valid)[:, 1]
    fold_auc = roc_auc_score(y_valid, fold_scores)
    importance_fold_auc.append(float(fold_auc))
    if not np.isclose(
        fold_auc, importance_nested["outer_auc_by_fold"][fold_number], rtol=0, atol=1e-12
    ):
        raise RuntimeError("A AUC reconstruída divergiu da validação aninhada.")

    result = permutation_importance(
        importance_pipeline,
        X_valid,
        y_valid,
        scoring="roc_auc",
        n_repeats=10,
        random_state=700 + fold_number,
        n_jobs=1,
    )
    importance_by_fold.append(result.importances_mean)

importance_matrix = np.vstack(importance_by_fold)
importance_table = pd.DataFrame(
    {
        "Variável": X_dev.columns,
        "Queda média de AUC": importance_matrix.mean(axis=0),
        "DP entre folds": importance_matrix.std(axis=0, ddof=1),
        "Folds com importância positiva": (importance_matrix > 0).sum(axis=0),
    }
).sort_values("Queda média de AUC", ascending=False).reset_index(drop=True)
display(importance_table.round(4))

# Visualizar as 15 maiores importâncias médias; barras de erro mostram DP entre folds.
plot_importance = importance_table.head(15).sort_values(
    "Queda média de AUC", ascending=True
)
fig, ax = plt.subplots(figsize=(8.5, 6.2))
ax.barh(
    plot_importance["Variável"],
    plot_importance["Queda média de AUC"],
    xerr=plot_importance["DP entre folds"],
    color="#4472C4",
    alpha=0.85,
    capsize=3,
)
ax.axvline(0, color="gray", linewidth=1)
ax.set_xlabel("Queda média da ROC-AUC após permutação")
ax.set_title("Permutation Importance — Random Forest (folds externos)")
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
importance_plot_path = PROJECT_ROOT / "results" / "stage6_permutation_importance.png"
fig.savefig(importance_plot_path, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

stage6_results = {
    "method": "permutation importance on outer validation folds",
    "model": importance_model_name,
    "metric": "ROC-AUC",
    "permutations_per_fold": 10,
    "development_outer_fold_auc_reconstruction": importance_fold_auc,
    "importance_by_feature": [
        {
            "feature": row["Variável"],
            "mean_auc_decrease": float(row["Queda média de AUC"]),
            "std_across_folds": float(row["DP entre folds"]),
            "positive_folds": int(row["Folds com importância positiva"]),
        }
        for _, row in importance_table.iterrows()
    ],
    "interpretation_limit": "predictive association only; not causal; correlated features can share or mask importance",
    "plot": "stage6_permutation_importance.png",
}
stage6_path = PROJECT_ROOT / "results" / "stage6_permutation_importance.json"
stage6_path.write_text(
    json.dumps(stage6_results, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Importâncias agregadas salvas em: {stage6_path}")


A importância por permutação resume a dependência preditiva do modelo selecionado nos folds externos. Não é efeito causal; variáveis correlacionadas podem compartilhar ou mascarar importância. Interprete os valores gerados pela execução atual, sem extrapolação clínica.
